# Machine Learning: Time-Series Forecasting
This notebook outlines training a Random Forest Regressor to forecast tomorrow's air quality index (AQI) and future energy demands.


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt

DATA_DIR = '../data'


## Load Feature Data


In [ ]:
weather = pd.read_csv(os.path.join(DATA_DIR, 'weather.csv'))
pollution = pd.read_csv(os.path.join(DATA_DIR, 'pollution.csv'))
df = pd.merge(weather, pollution, on=['Date', 'City'])
print('Datasets merged. Shape:', df.shape)


## Feature Engineering


In [ ]:
# Create date variables
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['AQI_Lag_1'] = df.groupby('City')['AQI'].shift(1)
# Drop rows with NaN due to lag
df.dropna(inplace=True)


## Model Training: Predicting AQI


In [ ]:
features = ['Temperature', 'Humidity', 'Rainfall', 'Wind_Speed', 'Month', 'DayOfWeek', 'AQI_Lag_1']
X = df[df['City'] == 'Mumbai'][features]
y = df[df['City'] == 'Mumbai']['AQI']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=50, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('Model Evaluation Metrics:')
print('Mean Absolute Error (MAE):', round(mean_absolute_error(y_test, y_pred), 3))
print('R2 Score (Variance Explained):', round(r2_score(y_test, y_pred), 3))


## Plotting Actual vs Predicted AQI


In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(y_test.values[:50], label='Actual AQI', color='blue')
plt.plot(y_pred[:50], label='Predicted AQI', color='red', linestyle='--')
plt.title('Actual vs Predicted AQI - Mumbai')
plt.legend()
plt.show()
